# 3. Metrics & Evaluation Charts


In [1]:
import pandas as pd
import pickle
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

os.makedirs('charts', exist_ok=True)

df = pd.read_csv('data/processed/processed_data.csv')
X = df.drop(columns=['Target'])

with open('models/label_encoder.pkl', 'rb') as f: le = pickle.load(f)
y_encoded = le.transform(df['Target'])

_, X_test, _, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

models = ['decision_tree', 'random_forest', 'xgboost']
metrics = {}

for model_name in models:
    with open(f'models/{model_name}.pkl', 'rb') as f:
        model = pickle.load(f)
    
    preds = model.predict(X_test)
    acc = float(accuracy_score(y_test, preds))
    metrics[model_name] = acc
    
    # Heatmap setup
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'{model_name.replace("_", " ").title()} Confusion Matrix')
    plt.ylabel('True Class')
    plt.xlabel('Predicted Class')
    plt.tight_layout()
    plt.savefig(f'charts/{model_name}_confusion_matrix.png', dpi=150)
    plt.close()

# Accuracy Bar Chart
pd.Series(metrics).plot(kind='bar', color='#4CAF50', edgecolor='black')
plt.title('Performance Comparison')
plt.ylabel('Accuracy Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('charts/model_performance_comparison.png', dpi=150)
plt.close()

with open('models/performance_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print('Evaluations completed! Beautiful charts exported to ./charts/')

/home/ashutosh/miniconda3/envs/ml-env/lib/python3.11/site-packages/xgboost/core.py:751: UserWarning: [06:12:57] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Evaluations completed! Beautiful charts exported to ./charts/
